# Fundamentación Metodológica y Preprocesamiento de Datos de Recaudo Tributario

**Documento de Trabajo - Nivel Maestría**

Este cuaderno documenta y justifica rigurosamente el pipeline de preprocesamiento aplicado al conjunto de datos de recaudo de rentas (`BaseRentasCedidasVF.xlsx`). El objetivo principal de este documento es proveer una sustentación técnica y académica sobre las decisiones de limpieza, transformación y, fundamentalmente, el truncamiento muestral (sample truncation) realizado sobre los periodos afectados por el choque exógeno de la pandemia de COVID-19.

El conjunto de datos resultante (`BaseRentasCedidasVF.xlsx`) constituye la base empírica depurada para el posterior modelado econométrico y pronóstico de series de tiempo.

## 1. Justificación Teórica del Truncamiento Muestral (2020 - Septiembre 2021)

### 1.1. El Choque Exógeno de la Pandemia COVID-19 como Ruptura Estructural

En el análisis de series de tiempo macroeconómicas y fiscales, la pandemia de COVID-19 representa un **choque exógeno sin precedentes** que indujo una volatilidad extrema y cambios abruptos en el nivel y la tendencia de las variables de recaudo. Las medidas de confinamiento estricto (lockdowns), la paralización de la actividad económica y las posteriores políticas de reactivación asimétrica generaron lo que en econometría se conoce como una **ruptura estructural severa (structural break)**.

De acuerdo con la literatura clásica de series de tiempo (Perron, 1989), la presencia de rupturas estructurales no modeladas sesga las pruebas de raíz unitaria hacia la no-estacionariedad y distorsiona severamente los estimadores autorregresivos (AR) y de medias móviles (MA). 

### 1.2. Implicaciones para el Pronóstico (Forecasting)

La inclusión de los datos correspondientes al año 2020 y los primeros tres trimestres de 2021 introduce **heterocedasticidad condicional** y **outliers aditivos (AO)** masivos en la serie. Como señalan Foroni, Marcellino y Stevanovic (2020) en su análisis sobre pronósticos durante la recesión del COVID-19, los modelos lineales estándar (como ARIMA o SARIMA) entrenados con datos que incluyen el choque pandémico tienden a generar pronósticos altamente inestables y con intervalos de confianza desproporcionadamente amplios, debido a que el modelo intenta "aprender" una dinámica de crisis que no es representativa del comportamiento estructural de largo plazo del recaudo.

### 1.3. Decisión Metodológica: Truncamiento de la Muestra

Para mitigar este sesgo y garantizar la robustez de los pronósticos futuros, se optó metodológicamente por un **truncamiento de la muestra por la izquierda (left-truncation)**. Se eliminaron todas las observaciones desde enero de 2020 hasta septiembre de 2021. 

Esta decisión se justifica bajo el supuesto de que la dinámica generadora de datos (Data Generating Process - DGP) post-pandemia (a partir del último trimestre de 2021) ha convergido hacia un "nuevo normal" estacionario o con una tendencia determinística estable, libre del ruido extremo de los confinamientos. Modelar a partir de octubre de 2021 asegura que los parámetros estimados reflejen las condiciones económicas contemporáneas y no las anomalías transitorias de la crisis sanitaria.

## 2. Pipeline de Preprocesamiento y Limpieza de la Base de Datos Original

A continuación, se documentan exhaustivamente las acciones técnicas ejecutadas sobre la base de datos original (`BaseRentasCedidasVF.xlsx`) para transformarla en la base final depurada. Cada paso responde a un criterio de calidad de datos (Data Quality) necesario para el modelado econométrico.

### 2.1. Ingesta y Verificación de Integridad Estructural
El proceso inició con la lectura del archivo original, el cual constaba de **231,986 observaciones** (filas) y 13 variables (columnas). Se verificó que la variable temporal clave (`FechaRecaudo`) estuviera presente para permitir el ordenamiento cronológico.

### 2.2. Ordenamiento Cronológico Estricto
Las series de tiempo requieren un ordenamiento temporal estricto para evitar la autocorrelación espuria y garantizar que la secuencia de eventos refleje la verdadera dinámica generadora de datos. Se forzó la conversión de la variable `FechaRecaudo` a formato `datetime64[ns]` y se ordenó el dataset de forma ascendente.

### 2.3. Ejecución del Truncamiento Muestral (Filtro COVID-19)
Como se justificó en la Sección 1, se procedió a aislar el choque pandémico mediante la eliminación de los registros correspondientes a los periodos de mayor distorsión económica y fiscal:
*   **Filtro 1 (Año 2020):** Se eliminaron **54,586 observaciones** correspondientes al año 2020, periodo caracterizado por los confinamientos estrictos y la paralización económica inicial.
*   **Filtro 2 (Enero - Septiembre 2021):** Se eliminaron **27,752 observaciones** adicionales correspondientes a los primeros tres trimestres de 2021, periodo de reactivación asimétrica y alta volatilidad.
*   **Resultado del Truncamiento:** La base de datos se redujo a **149,648 observaciones**, estableciendo el inicio de la serie temporal en octubre de 2021.

In [ ]:
import pandas as pd
import numpy as np

# 1. Carga de Datos Originales
# file_path = 'BaseRentasCedidasVF.xlsx'
# df = pd.read_excel(file_path)
# print(f"Total de filas iniciales: {len(df)}") # 231,986 filas

# 2. Ordenamiento Cronológico
# df['FechaRecaudo'] = pd.to_datetime(df['FechaRecaudo'], errors='coerce')
# df_sorted = df.sort_values(by='FechaRecaudo', ascending=True)

# 3. Truncamiento Muestral (Eliminación del Choque COVID-19)
# 3.1. Eliminación del año 2020
# df_filtered = df_sorted[df_sorted['FechaRecaudo'].dt.year != 2020]
# Filas eliminadas: 54,586

# 3.2. Eliminación de Enero a Septiembre de 2021
# condicion_eliminar = (df_filtered['FechaRecaudo'].dt.year == 2021) & (df_filtered['FechaRecaudo'].dt.month <= 9)
# df_filtered = df_filtered[~condicion_eliminar]
# Filas eliminadas: 27,752
# Total de filas restantes tras truncamiento: 149,648

## 3. Control de Calidad de Datos (Data Quality Assurance)

Una vez truncada la muestra para aislar el choque pandémico, se procedió a un riguroso control de calidad de los datos restantes (octubre 2021 en adelante) para asegurar la integridad del análisis econométrico.

### 3.1. Tratamiento de Duplicados Exactos
Se identificaron **238 registros** que eran copias exactas en todas sus dimensiones (fecha, NIT, valor, concepto). Estos duplicados introducen un sesgo al alza en la estimación del recaudo total y fueron eliminados, reduciendo la base a **149,410 observaciones**.

### 3.2. Análisis de Valores Nulos (Missing Values)
Se verificó la completitud de la base de datos. El resultado arrojó **0 valores nulos** en todas las columnas, lo que indica una completitud del 100% y elimina la necesidad de técnicas de imputación (ej. KNN, MICE).

### 3.3. Limpieza de Cadenas de Texto (String Stripping)
Para evitar errores de agrupación categórica en análisis posteriores, se eliminaron espacios en blanco al inicio y final de todas las variables categóricas (tipo `object`).

### 3.4. Conversión de Tipos de Datos (Data Type Casting)
La variable dependiente principal, `ValorRecaudo`, presentaba un formato de cadena de texto (string) debido a la presencia de comas como separadores decimales. Se realizó un casting a formato numérico (`float64`) para permitir operaciones aritméticas y agregaciones.

### 3.5. Identificación de Valores Atípicos Negativos
Durante el análisis descriptivo de la variable `ValorRecaudo`, se detectaron **26 registros con valores negativos** (el valor mínimo registrado fue de -$483.6 millones). Estos valores, que presumiblemente corresponden a devoluciones tributarias o ajustes contables, requieren un tratamiento especial (ej. imputación a cero o modelado separado) durante la fase de agregación mensual para evitar distorsiones en la serie de tiempo agregada.

### 3.6. Exportación del Dataset Final
El dataset final, limpio, ordenado y truncado, se exportó como `BaseRentasCedidasVF.xlsx` con un total de **149,410 observaciones**.

In [ ]:
# 4. Tratamiento de Duplicados Exactos
# df_filtered = df_filtered.drop_duplicates()
# Filas restantes: 149,410

# 5. Análisis de Valores Nulos (Missing Values)
# nulos_por_columna = df_filtered.isnull().sum()
# Resultado: 0 valores nulos en todas las columnas.

# 6. Limpieza de Cadenas de Texto (String Stripping)
# columnas_texto = df_filtered.select_dtypes(include=['object']).columns
# for col in columnas_texto:
#     df_filtered[col] = df_filtered[col].astype(str).str.strip()

# 7. Conversión de Tipos de Datos (Data Type Casting)
# df_filtered['ValorRecaudo'] = df_filtered['ValorRecaudo'].astype(str).str.replace(',', '.')
# df_filtered['ValorRecaudo'] = pd.to_numeric(df_filtered['ValorRecaudo'], errors='coerce').fillna(0)

# 8. Identificación de Valores Atípicos Negativos
# valores_negativos = df_filtered[df_filtered['ValorRecaudo'] < 0]
# Resultado: 26 registros con valores negativos.

# 9. Exportación del Dataset Final
# output_path = 'BaseRentasCedidasVF.xlsx'
# df_filtered.to_excel(output_path, index=False)
# Total de filas finales: 149,410

## 4. Referencias Bibliográficas

*   **Foroni, C., Marcellino, M., & Stevanovic, D. (2020).** *Forecasting the Covid-19 recession and recovery: Lessons from the financial crisis*. International Journal of Forecasting, 36(4), 1126-1148. (Justificación del impacto de crisis extremas en modelos de pronóstico).
*   **Perron, P. (1989).** *The great crash, the oil price shock, and the unit root hypothesis*. Econometrica, 57(6), 1361-1401. (Teoría fundamental sobre rupturas estructurales y estacionariedad).
*   **Lütkepohl, H., & Krätzig, M. (2004).** *Applied time series econometrics*. Cambridge university press. (Metodología general para el tratamiento de outliers y cambios de régimen).